<a href="https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/master/tutorials/06_topology_optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 6: Generative Inverse Design -- Topology Optimization

**The Pain Point:** Evaluating existing materials is slow; inventing the mathematically optimal material structure is nearly impossible with traditional tools.

**The Solution:** We put OpenImpala inside a "Hill Climbing" optimization loop. The algorithm autonomously swaps solid and pore voxels (keeping porosity strictly fixed), querying OpenImpala every step to carve out the perfect pore network.

**In this tutorial we will:**
1. Start with a completely random 50/50 microstructure.
2. Iteratively swap solid and pore voxels to improve (lower) the tortuosity.
3. Plot the optimization curve showing convergence.
4. Visualize the resulting optimized structure.

By strictly swapping voxels, we guarantee that the overall porosity stays exactly at 50%, forcing the algorithm to find the most efficient possible pathways for that specific volume fraction. This proves OpenImpala is stable and fast enough to be used as a cost-function evaluator inside generative AI design loops.

In [ ]:
# Install OpenImpala and visualization libraries
!pip install -q openimpala matplotlib

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import openimpala as oi

print(f"OpenImpala version {oi.__version__} loaded.")

SIZE = 16  # Keep domain small so the tutorial runs in < 1 minute
ITERATIONS = 50

# 1. Initialize a completely random 50/50 microstructure
np.random.seed(42)
micro = np.random.choice([0, 1], size=(SIZE, SIZE, SIZE)).astype(np.int32)

In [ ]:
best_tau = np.inf
tau_history = []

print("Starting Topology Optimization...")
t_start = time.time()

with oi.Session():
    # Evaluate initial state
    perc = oi.percolation_check(micro, phase=1, direction="z")
    if perc.percolates:
        best_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity

    print(f"Initial Tortuosity: {best_tau:.4f}")
    tau_history.append(best_tau)

    for i in range(ITERATIONS):
        # Find all current pore (1) and solid (0) coordinates
        pore_coords = np.argwhere(micro == 1)
        solid_coords = np.argwhere(micro == 0)

        # Pick one random pore and one random solid
        p_idx = pore_coords[np.random.choice(len(pore_coords))]
        s_idx = solid_coords[np.random.choice(len(solid_coords))]

        # Mutate: Swap them (this guarantees VF stays exactly the same)
        micro[tuple(p_idx)] = 0
        micro[tuple(s_idx)] = 1

        # Evaluate new physics
        perc = oi.percolation_check(micro, phase=1, direction="z")
        new_tau = np.inf
        if perc.percolates:
            new_tau = oi.tortuosity(micro, phase=1, direction="z").tortuosity

        # Hill Climbing logic: Only keep the swap if it improved transport!
        if new_tau < best_tau:
            best_tau = new_tau
            print(f"Iteration {i+1:02d} | SUCCESS | New Best Tau: {best_tau:.4f}")
        else:
            # Revert the mutation
            micro[tuple(p_idx)] = 1
            micro[tuple(s_idx)] = 0

        tau_history.append(best_tau)

t_total = time.time() - t_start
print(f"\nOptimization complete in {t_total:.2f} seconds.")
print(f"Final Tortuosity: {best_tau:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=120)

# --- Plot 1: Optimization curve ---
ax1.plot(tau_history, 'g-', lw=2, marker='o', markersize=4)
ax1.set_title("Topology Optimization (Fixed Porosity 50%)", fontweight='bold')
ax1.set_xlabel("Mutation Iteration", fontweight='bold')
ax1.set_ylabel("Tortuosity (Lower is Better)", fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# --- Plot 2: Optimized slice ---
ax2.imshow(micro[:, SIZE//2, :], cmap='gray')
ax2.set_title("Optimized Microstructure Slice", fontweight='bold')
ax2.axis('off')

plt.tight_layout()
plt.show()

## What's Next?

You just demonstrated that OpenImpala is fast and stable enough to serve as a physics evaluator inside an optimization loop. This is the foundation for more sophisticated generative design approaches (genetic algorithms, gradient-based topology optimization, reinforcement learning).

**Continue the journey:**
- [Tutorial 7: Laptop to Supercomputer](07_hpc_scaling.ipynb) -- Scale up to massive datasets with MPI.
- [Tutorial 1: Zero to Physics](01_hello_openimpala.ipynb) -- Back to basics with synthetic microstructures.